# 🚀 Preprocessing: Dal Testo Grezzo alla Qualità

**Prima di trasformare il testo in numeri, dobbiamo pulirlo, normalizzarlo e portarlo in qualità.** Senza questa fase, il rumore nei dati ridurrebbe drasticamente l'efficacia dei nostri modelli (GIGO: *Garbage In, Garbage Out*).

---

# 1) Pulizia del Testo
L'obiettivo è eliminare gli elementi che non aggiungono valore semantico al task specifico e che aumentano inutilmente la dimensione del vocabolario.

## 1.1) Rimozione di Rumore (HTML, URL, Tag, Hashtag, Emoji)
Tutto ciò che è "codice" o metadato spesso non serve per l'analisi del significato profondo.
* **Input:** `"Visita il sito <a href='https://ai.com'>QUI</a> per info! #NLP 🚀"`
* **Output:** `"Visita il sito QUI per info! NLP"`

## 1.2) Rimozione di punteggiatura e numeri
In molti task (come la sentiment analysis), la punteggiatura non aiuta, anche se in altri contesti (come la traduzione automatica) può essere fondamentale.
* **Input:** `"L'intelligenza artificiale nel 2026? Un passo avanti!"`
* **Output:** `"L intelligenza artificiale nel Un passo avanti"`

## 1.3) Conversione in minuscolo (Lowercasing)
Serve per uniformare le parole ed evitare che il modello tratti "Gatto" e "gatto" come due entità distinte.
* **Input:** `"Il GATTO insegue il Topo."`
* **Output:** `"il gatto insegue il topo."`

---

# 2) Tokenizzazione
È il processo di scomposizione del testo in unità più piccole chiamate **token**. Possono essere parole, frasi o sotto-parole.

| Tipo | Esempio di Input | Token Risultanti |
| :--- | :--- | :--- |
| **Parola** | `"Il NLP è potente"` | `["Il", "NLP", "è", "potente"]` |
| **Frase** | `"Ciao! Come stai?"` | `["Ciao!", "Come stai?"]` |
| **Sub-word** | `"Incredibilmente"` | `["Incredibil", "mente"]` |

---

# 3) Normalizzazione
Portiamo le parole a una forma standard per ridurre la complessità del vocabolario.

## 3.1) Rimozione delle Stop Word
Eliminiamo le "parole vuote" (articoli, preposizioni, congiunzioni) che appaiono molto spesso ma portano poco significato semantico.
* **Input:** `"Il libro è sul tavolo"`
* **Filtrato:** `["libro", "tavolo"]`

## 3.2) Stemming
Ricondurre le parole alla propria radice (stem) attraverso regole rigide di troncamento. È veloce ma può generare radici non esistenti nel dizionario.
* **Esempio:**
    * `"correndo"`, `"corsi"`, `"corridore"` $\rightarrow$ **"corr"**
    * `"università"`, `"universitario"` $\rightarrow$ **"univers"**

## 3.3) Lemmatizzazione
Ricondurre le parole alla loro forma base (lemma) usando un dizionario e l'analisi morfologica. È molto più preciso dello stemming perché considera il contesto (POS tagging).
* **Esempio:**
    * `"migliore"` $\rightarrow$ **"buono"**
    * `"andavamo"` $\rightarrow$ **"andare"**
    * `"uomini"` $\rightarrow$ **"uomo"**

---

# Passiamo alla pratica

## Import delle librerie

In [1]:
import re
import string
import nltk

# Scarica le risorse necessarie
nltk.download('punkt')
nltk.download('punkt_tab') # Aggiunto per le versioni più recenti di NLTK
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

## Consideriamo questo testo

In [2]:
testo_originale = """
L'inflazione nel 2026 è in forte AUMENTO!!! 📈
Prezzi alle stelle su: [https://finanza.it/report](https://finanza.it/report).
#Economia #Mercati... Stiamo monitorando la situazione.
"""
print("ORIGINALE:", testo_originale)

ORIGINALE: 
L'inflazione nel 2026 è in forte AUMENTO!!! 📈 
Prezzi alle stelle su: [https://finanza.it/report](https://finanza.it/report). 
#Economia #Mercati... Stiamo monitorando la situazione.



## 1) Pulizia

In [3]:
def clean_text(text):
    # Rimozione di apostrofi e caratteri speciali
    text = text.replace("'", " ").replace("’", " ")
    # Rimozione HTML e URL
    text = re.sub(r'<.*?>', '', text) # Rimuove tag HTML
    text = re.sub(r'http\S+', '', text) # Rimuove URL

    # Rimozione Hashtag ed Emoji (semplificato)
    text = re.sub(r'#\S+', '', text) # Rimuove hashtag
    text = text.encode('ascii', 'ignore').decode('ascii') # Rimuove emoji
    # Conversione in minuscolo
    text = text.lower()

    # Rimozione punteggiatura e numeri
    # Creiamo una regex che tiene solo le lettere
    text = re.sub(r'[^a-zàèìòù\s]', '', text)

    return text.strip()

text_cleaned = clean_text(testo_originale)
print(f"Dopo la Pulizia:\n{text_cleaned}")

Dopo la Pulizia:
l inflazione nel   in forte aumento  
prezzi alle stelle su  
  stiamo monitorando la situazione


In [4]:
from nltk.tokenize import word_tokenize

tokens = word_tokenize(text_cleaned)
print(f"Token estratti:\n{tokens}")

Token estratti:
['l', 'inflazione', 'nel', 'in', 'forte', 'aumento', 'prezzi', 'alle', 'stelle', 'su', 'stiamo', 'monitorando', 'la', 'situazione']


In [26]:
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
from nltk.stem import WordNetLemmatizer

#Rimozione Stop Words (Italiano)
stop_words = set(stopwords.words('italian')) #Potete selezionare un dizionario specifico per l'italiano
tokens_filtered = [word for word in tokens if word not in stop_words]

#Stemming (Ricondurre alla radice)
stemmer = SnowballStemmer('italian')
tokens_stemmed = [stemmer.stem(word) for word in tokens_filtered]

#Lemmatizzazione
# Nota: La lemmatizzazione accurata in italiano richiede librerie come Spacy.
# Qui usiamo WordNet (principalmente inglese) per mostrare il concetto.
lemmatizer = WordNetLemmatizer()
tokens_lemmatized = [lemmatizer.lemmatize(word) for word in tokens_filtered]

print(f"Senza Stop Words: {tokens_filtered}")
print(f"Dopo Stemming:    {tokens_stemmed}")
print(f"Dopo Lemmatizing: {tokens_lemmatized}")

Senza Stop Words: ['inflazione', 'forte', 'aumento', 'prezzi', 'stelle', 'monitorando', 'situazione']
Dopo Stemming:    ['inflazion', 'fort', 'aument', 'prezz', 'stell', 'monitor', 'situazion']
Dopo Lemmatizing: ['inflazione', 'forte', 'aumento', 'prezzi', 'stelle', 'monitorando', 'situazione']


# Esercizio:

In [27]:
frasi = ["Segui la diretta su <https://live.nlp.it> per scoprire l'IA! #Innovazione 🚀",
         "The artificial intelligence is changing the way we live and work in the 21st century!",
         "I programmatori programmano programmi programmabili con la programmazione."]

#reminder: l'accesso agli elementi di una lista avviene tramite indice, partendo da 0


In [ ]:
# Query 1 (Pulizia): "Rimuovi URL e Hashtag dalla prima frase nella lista, converti il testo in minuscolo e mantieni solo i caratteri alfabetici (escludendo le emoji)."
# risultato atteso: "segui la diretta su per scoprire l ia"

In [28]:
frase_pulita = clean_text(frasi[0])
print(f"Dopo la Pulizia:\n{frase_pulita}")



Dopo la Pulizia:
segui la diretta su  per scoprire l ia


In [ ]:
# Query 2 (Stopword - EN): "Dividi la Frase B in token e rimuovi tutte le stopword della lingua inglese. Rimuovi anche la punteggiatura e i numeri."
# risultato atteso: ['artificial', 'intelligence', 'changing', 'way', 'live', 'work', 'century']

In [32]:
frase_2_pulita = clean_text(frasi[1])
tokens_frase_2 = word_tokenize(frase_2_pulita)

#Rimozione Stop Words (Inglese)
stop_word = set(stopwords.words('english')) # dizionario inglese
tokens_puliti = [word for word in tokens_frase_2 if word not in stop_word]
print(f"Senza Stop Words: {tokens_puliti}")


Senza Stop Words: ['artificial', 'intelligence', 'changing', 'way', 'live', 'work', 'st', 'century']


In [ ]:
#Query 3 (Riduzione - IT): "Tokenizza la Frase C e applica lo Stemming (usando lo SnowballStemmer per l'italiano) per ridurre ogni parola alla propria radice."
#risultato atteso: ['programm', 'programm', 'programm', 'programm', 'con', 'la', 'programm']

In [34]:
frase_3_pulita = clean_text(frasi[2])
tokens_frase_3 = word_tokenize(frase_3_pulita)

#Stemming (Ricondurre alla radice)
stemmare = SnowballStemmer('italian')
tokens_stemmati = [stemmare.stem(word) for word in tokens_frase_3]

print(f"Dopo Stemming:    {tokens_stemmati}")

Dopo Stemming:    ['i', 'programm', 'programm', 'programm', 'programm', 'con', 'la', 'programm']
